In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
adata_v5 = sc.read_h5ad("source_files/adata_combined.h5ad")

In [ ]:
adata_combined = adata_v5

In [ ]:
# WT cells from the two tracking days
adata_wt = adata_combined[adata_combined.obs['sample'].isin(['WT_d6', 'WT_d9'])].copy()

# start with clones seen in PSC/Epi at WT_d6
adata_psc_epi_d6 = adata_wt[(adata_wt.obs['sample'] == 'WT_d6') & (adata_wt.obs['celltype_v2'] == 'PSC/Epi')].copy()
psc_epi_d6_clones = adata_psc_epi_d6.obs['larry_clone'].unique()

print(f"Number of unique clones in WT_d6 PSC/Epi cells: {len(psc_epi_d6_clones)}")
print(f"Total PSC/Epi cells in WT_d6: {adata_psc_epi_d6.shape[0]}")

# then keep clones still present at WT_d9, in any cell type
clone_sample_presence = adata_wt.obs[adata_wt.obs['larry_clone'].isin(psc_epi_d6_clones)].groupby('larry_clone')['sample'].unique()
persistent_psc_origin_clones = [clone for clone, samples in clone_sample_presence.items() 
                               if ('WT_d6' in samples) and ('WT_d9' in samples)]

print(f"\nNumber of PSC/Epi-origin clones that persist to WT_d9: {len(persistent_psc_origin_clones)}")

if len(persistent_psc_origin_clones) > 0:
    # rank the persistent clones by abundance
    persistent_clone_counts = adata_wt.obs[adata_wt.obs['larry_clone'].isin(persistent_psc_origin_clones)]['larry_clone'].value_counts()
    
    # at most 29 clones are shown
    n_clones_to_show = min(29, len(persistent_clone_counts))
    top_persistent_clones = persistent_clone_counts.head(n_clones_to_show).index.tolist()

In [ ]:
# write the tracking labels back onto the full object
# everything starts as Other
adata_combined.obs['clone_tracking'] = 'Other'

# d6 PSC/Epi cells from clones that make it to d9
psc_start_mask = (
    (adata_combined.obs['sample'] == 'WT_d6') & 
    (adata_combined.obs['celltype_v2'] == 'PSC/Epi') & 
    (adata_combined.obs['larry_clone'].isin(top_persistent_clones))
)
adata_combined.obs.loc[psc_start_mask, 'clone_tracking'] = 'PSC/Epi-Start'

# matching cells at d9
end_mask = (
    (adata_combined.obs['sample'] == 'WT_d9') & 
    (adata_combined.obs['larry_clone'].isin(top_persistent_clones))
)
adata_combined.obs.loc[end_mask, 'clone_tracking'] = 'End'

print(f"\nTracking summary:")
print(adata_combined.obs['clone_tracking'].value_counts())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# WT_d6 Start tracking cells colored by celltype_v2
# =========================================================

# colors used for the cell types
palette = [
    "#6BAED6", "#4DB6AC", "#74C476", "#A1D99B",
    "#FDD835", "#FDAE6B", "#F46D43", "#E53935",
    "#D81B60", "#8E24AA", "#9575CD", "#BDBDBD"
]

# keep this order fixed so colors dont jump between panels
adata_combined.obs["celltype_v2"] = adata_combined.obs["celltype_v2"].astype("category")
celltype_order = list(adata_combined.obs["celltype_v2"].cat.categories)

# line the palette up with that order
adata_combined.uns["celltype_v2_colors"] = palette[:len(celltype_order)]
ct_color_map = dict(zip(celltype_order, adata_combined.uns["celltype_v2_colors"]))

# WT_d6 cells drawn on top
mask_start = (
    (adata_combined.obs["clone_tracking"] == "PSC/Epi-Start") &
    (adata_combined.obs["sample"] == "WT_d6")
)

# UMAP coordinates from the full object
umap = adata_combined.obsm["X_umap"]

# colors for the starting cells
start_ct = adata_combined.obs.loc[mask_start, "celltype_v2"].astype(str)
start_colors = start_ct.map(ct_color_map).values

# plot
plt.figure(figsize=(10, 10))

# all cells in the background
plt.scatter(
    umap[:, 0], umap[:, 1],
    c="#D3D3D3", s=6, alpha=0.25,
    edgecolors="none", zorder=1
)

# starting cells go on top, colored by cluster
plt.scatter(
    umap[mask_start, 0], umap[mask_start, 1],
    c=start_colors,
    s=600,
    edgecolors="black",
    linewidths=2.2,
    alpha=1.0,
    zorder=3
)

# no axes for this panel
ax = plt.gca()
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel(""); ax.set_ylabel("")
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig("WT_d6_Start_tracking_byCelltypeV2.tiff",
            dpi=600, format="tiff", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# WT_d9 End tracking cells colored by celltype_v2
# keeps background grey, only highlighted cells are colored
# =========================================================

# same cell-type colors
palette = [
    "#6BAED6", "#4DB6AC", "#74C476", "#A1D99B",
    "#FDD835", "#FDAE6B", "#F46D43", "#E53935",
    "#D81B60", "#8E24AA", "#9575CD", "#BDBDBD"
]

# keep category order fixed or the palette moves around
adata_combined.obs["celltype_v2"] = adata_combined.obs["celltype_v2"].astype("category")
celltype_order = list(adata_combined.obs["celltype_v2"].cat.categories)

# save colors on the AnnData object in category order
adata_combined.uns["celltype_v2_colors"] = palette[:len(celltype_order)]
ct_color_map = dict(zip(celltype_order, adata_combined.uns["celltype_v2_colors"]))

# WT_d9 cells drawn on top
mask_end = (
    (adata_combined.obs["clone_tracking"] == "End") &
    (adata_combined.obs["sample"] == "WT_d9")
)

# same UMAP coordinates
umap = adata_combined.obsm["X_umap"]

# colors for the d9 cells
end_ct = adata_combined.obs.loc[mask_end, "celltype_v2"].astype(str)
end_colors = end_ct.map(ct_color_map).values

# plot
plt.figure(figsize=(10, 10))

# background: all cells in light grey
plt.scatter(
    umap[:, 0], umap[:, 1],
    c="#D3D3D3", s=6, alpha=0.25,
    edgecolors="none", zorder=1
)

# overlay: End cells colored by cluster
plt.scatter(
    umap[mask_end, 0], umap[mask_end, 1],
    c=end_colors,
    s=600,
    edgecolors="black",
    linewidths=2.2,
    alpha=1.0,
    zorder=3
)

# clean axes
ax = plt.gca()
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel(""); ax.set_ylabel("")
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig("WT_d9_End_tracking_byCelltypeV2.tiff",
            dpi=600, format="tiff", bbox_inches="tight")
plt.show()

In [ ]:
# same tracking for the KO group


# KO cells from the two tracking days
adata_ko = adata_combined[adata_combined.obs['sample'].isin(['KO_d6', 'KO_d9'])].copy()

# start with clones seen in PSC/Epi at KO_d6
adata_psc_epi_d6 = adata_ko[(adata_ko.obs['sample'] == 'KO_d6') & (adata_ko.obs['celltype_v2'] == 'PSC/Epi')].copy()
psc_epi_d6_clones = adata_psc_epi_d6.obs['larry_clone'].unique()

print(f"Number of unique clones in KO_d6 PSC/Epi cells: {len(psc_epi_d6_clones)}")
print(f"Total PSC/Epi cells in KO_d6: {adata_psc_epi_d6.shape[0]}")

# then keep clones still present at KO_d9, in any cell type
clone_sample_presence = adata_ko.obs[adata_ko.obs['larry_clone'].isin(psc_epi_d6_clones)].groupby('larry_clone')['sample'].unique()
persistent_psc_origin_clones = [clone for clone, samples in clone_sample_presence.items() 
                               if ('KO_d6' in samples) and ('KO_d9' in samples)]

print(f"\nNumber of PSC/Epi-origin clones that persist to KO_d9: {len(persistent_psc_origin_clones)}")

if len(persistent_psc_origin_clones) > 0:
    # rank the persistent clones by abundance
    persistent_clone_counts = adata_ko.obs[adata_ko.obs['larry_clone'].isin(persistent_psc_origin_clones)]['larry_clone'].value_counts()
    
    # at most 29 clones are shown
    n_clones_to_show = min(29, len(persistent_clone_counts))
    top_persistent_clones = persistent_clone_counts.head(n_clones_to_show).index.tolist()


In [ ]:
# write the KO tracking labels back onto the full object
# everything starts as Other
adata_combined.obs['clone_tracking'] = 'Other'

# d6 PSC/Epi cells from clones that make it to d9
psc_start_mask = (
    (adata_combined.obs['sample'] == 'KO_d6') & 
    (adata_combined.obs['celltype_v2'] == 'PSC/Epi') & 
    (adata_combined.obs['larry_clone'].isin(top_persistent_clones))
)
adata_combined.obs.loc[psc_start_mask, 'clone_tracking'] = 'PSC/Epi-Start'

# matching cells at d9
end_mask = (
    (adata_combined.obs['sample'] == 'KO_d9') & 
    (adata_combined.obs['larry_clone'].isin(top_persistent_clones))
)
adata_combined.obs.loc[end_mask, 'clone_tracking'] = 'End'

print(f"\nTracking summary:")
print(adata_combined.obs['clone_tracking'].value_counts())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# KO_d6 Start tracking cells colored by celltype_v2
# =========================================================

palette = [
    "#6BAED6", "#4DB6AC", "#74C476", "#A1D99B",
    "#FDD835", "#FDAE6B", "#F46D43", "#E53935",
    "#D81B60", "#8E24AA", "#9575CD", "#BDBDBD"
]

# keep the category order and colors together
adata_combined.obs["celltype_v2"] = adata_combined.obs["celltype_v2"].astype("category")
celltype_order = list(adata_combined.obs["celltype_v2"].cat.categories)
adata_combined.uns["celltype_v2_colors"] = palette[:len(celltype_order)]
ct_color_map = dict(zip(celltype_order, adata_combined.uns["celltype_v2_colors"]))

# KO_d6 cells drawn on top
mask_start = (
    (adata_combined.obs["clone_tracking"] == "PSC/Epi-Start") &
    (adata_combined.obs["sample"] == "KO_d6")
)

umap = adata_combined.obsm["X_umap"]

start_ct = adata_combined.obs.loc[mask_start, "celltype_v2"].astype(str)
start_colors = start_ct.map(ct_color_map).values

plt.figure(figsize=(10, 10))

# all cells in the background
plt.scatter(
    umap[:, 0], umap[:, 1],
    c="#D3D3D3", s=6, alpha=0.25,
    edgecolors="none", zorder=1
)

# starting cells on top, colored by cluster
plt.scatter(
    umap[mask_start, 0], umap[mask_start, 1],
    c=start_colors,
    s=600,
    edgecolors="black",
    linewidths=2.2,
    alpha=1.0,
    zorder=3
)

# no axes for this panel
ax = plt.gca()
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel(""); ax.set_ylabel("")
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig("KO_d6_Start_tracking_byCelltypeV2.tiff",
            dpi=600, format="tiff", bbox_inches="tight")
plt.show()

print("KO_d6 Start cells highlighted:", int(mask_start.sum()))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# KO_d9 End tracking cells colored by celltype_v2
# =========================================================

palette = [
"#6BAED6", "#4DB6AC", "#74C476", "#A1D99B",
"#FDD835", "#FDAE6B", "#F46D43", "#E53935",
"#D81B60", "#8E24AA", "#9575CD", "#BDBDBD"
]

# keep the same category order
adata_combined.obs["celltype_v2"] = adata_combined.obs["celltype_v2"].astype("category")
celltype_order = list(adata_combined.obs["celltype_v2"].cat.categories)

adata_combined.uns["celltype_v2_colors"] = palette[:len(celltype_order)]
ct_color_map = dict(zip(celltype_order, adata_combined.uns["celltype_v2_colors"]))

# KO_d9 cells drawn on top
mask_end = (
    (adata_combined.obs["clone_tracking"] == "End") &
    (adata_combined.obs["sample"] == "KO_d9")
)

umap = adata_combined.obsm["X_umap"]

end_ct = adata_combined.obs.loc[mask_end, "celltype_v2"].astype(str)
end_colors = end_ct.map(ct_color_map).values

plt.figure(figsize=(10, 10))

# all cells in the background
plt.scatter(
    umap[:, 0], umap[:, 1],
    c="#D3D3D3", s=6, alpha=0.25,
    edgecolors="none", zorder=1
)

# ending cells on top, colored by cluster
plt.scatter(
    umap[mask_end, 0], umap[mask_end, 1],
    c=end_colors,
    s=600,
    edgecolors="black",
    linewidths=2.2,
    alpha=1.0,
    zorder=3
)

# no axes for this panel
ax = plt.gca()
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel(""); ax.set_ylabel("")
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig("KO_d9_End_tracking_byCelltypeV2.tiff",
            dpi=600, format="tiff", bbox_inches="tight")
plt.show()

print("KO_d9 End cells highlighted:", int(mask_end.sum()))

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

# use exactly the same cluster colors for WT and KO
adata_wt.uns["celltype_v2_colors"] = adata_combined.uns["celltype_v2_colors"]
adata_ko.uns["celltype_v2_colors"] = adata_combined.uns["celltype_v2_colors"]

# cell-type palette
palette = [
    "#6BAED6", "#4DB6AC", "#74C476", "#A1D99B",
    "#FDD835", "#FDAE6B", "#F46D43", "#E53935",
    "#D81B60", "#8E24AA", "#9575CD", "#BDBDBD"
]
n_clusters = adata_combined.obs["celltype_v2"].nunique()
palette = palette[:n_clusters]

POINT_SIZE = 90

# WT
fig, ax = plt.subplots(figsize=(9, 10))  # panel size is set here
sc.pl.umap(
    adata_wt,
    color="celltype_v2",
    legend_loc=None,
    title=None,
    frameon=False,
    size=POINT_SIZE,
    palette=palette,
    ax=ax, 
    show=False
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("#B0B0B0")   # dark grey
    spine.set_linewidth(2)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig(
    "WT_UMAP_edit.tiff",
    format="tiff",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

# KO
fig, ax = plt.subplots(figsize=(9, 10))  # same panel size as WT
sc.pl.umap(
    adata_ko,
    color="celltype_v2",
    legend_loc=None,
    title=None,
    frameon=False,
    size=POINT_SIZE,
    palette=palette,
    ax=ax,
    show=False
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("#B0B0B0")   # dark grey
    spine.set_linewidth(2)

ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig(
    "KO_UMAP_edit.tiff",
    format="tiff",
    dpi=600,
    bbox_inches="tight"
)
plt.show()